In [1]:
import cv2
import numpy as np
import pandas as pd

# --- Ajustes para precisar alterar cor das marcações ---
# BLUE_LOWER = np.array([90, 100, 50])
# BLUE_UPPER = np.array([130, 255, 255])
ORANGE_LOWER = np.array([5, 120, 120])   # HSV limite baixo para laranja
ORANGE_UPPER = np.array([20, 255, 255])  # HSV limite alto para laranja
MIN_CONTOUR_AREA = 20                    # ignora pequenos defeitos na imagem
# ------------------------------------------------

def find_shot_centroids(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, ORANGE_LOWER, ORANGE_UPPER)
    # optional morphological clean
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    centers = []
    for c in contours:
        if cv2.contourArea(c) < MIN_CONTOUR_AREA:
            continue
        M = cv2.moments(c)
        if M['m00'] == 0: continue
        cx = int(M['m10']/M['m00'])
        cy = int(M['m01']/M['m00'])
        centers.append((cx, cy))
    return centers

def detect_court_corners(img):
    # Tenta detectar contorno da quadra como retângulo branco
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, th = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    # Escolhe o maior contorno
    best = None
    for c in contours:
        area = cv2.contourArea(c)
        if area < 10000: continue
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02*peri, True)
        if len(approx) == 4:
            best = approx.reshape(4,2)
            break
    return best  # None se não encontrado

def order_pts(pts):
    # ordem: esq-superior, dir-superior, dir-inferior, esq-inferior
    rect = np.zeros((4,2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def compute_homography(src_pts):
    # target é o retângulo, em pixels, correspondente a uma quadra real (50x94 ft)
    dst = np.array([[0,0],[50,0],[50,94],[0,94]], dtype="float32")
    H, _ = cv2.findHomography(src_pts, dst)
    return H

def pixels_to_court_coords(centers, H):
    pts = np.array(centers, dtype='float32').reshape(-1,1,2)
    mapped = cv2.perspectiveTransform(pts, H).reshape(-1,2)
    # convert to mplbasketball origin: center of court (X horizontal, Y vertical)
    # dst used: x in [0,50] left->right, y in [0,94] top->bottom
    X = mapped[:,0] - 47.0   # center at 0
    Y = mapped[:,1] - 25.0   # center at 0

    return np.column_stack([X, Y])

def get_corners_from_click(img, window_name="Clique os 4 cantos: TL, TR, BR, BL"):
    pts = []
    img_copy = img.copy()

    def onclick(event, x, y, flags, param):
        nonlocal img_copy
        if event == cv2.EVENT_LBUTTONDOWN:
            pts.append((x, y))
            # desenha círculo e número
            cv2.circle(img_copy, (x, y), 6, (0, 255, 0), -1)
            cv2.putText(img_copy, str(len(pts)), (x+8, y-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2, cv2.LINE_AA)
        elif event == cv2.EVENT_RBUTTONDOWN:
            # desfazer último clique (botão direito)
            if pts:
                pts.pop()
                img_copy = img.copy()
                for i, (px, py) in enumerate(pts, start=1):
                    cv2.circle(img_copy, (px, py), 6, (0,255,0), -1)
                    cv2.putText(img_copy, str(i), (px+8, py-8),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2, cv2.LINE_AA)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(window_name, onclick)

    instructions = ("Clique 4 pontos na ordem: TL, TR, BR, BL.\n"
                    "Botão esquerdo: adicionar; Botão direito: desfazer; ESC: cancelar")
    print(instructions)

    while True:
        cv2.imshow(window_name, img_copy)
        key = cv2.waitKey(1) & 0xFF
        # ESC para cancelar
        if key == 27:
            break
        if len(pts) >= 4:
            break

    cv2.destroyWindow(window_name)
    if len(pts) < 4:
        return None
    return np.array(pts[:4], dtype='float32')

# --- Main pipeline ---
def image_to_shot_dataframe(image_path, force_manual=True):
    """
    Se force_manual=True (padrão), o usuário será solicitado a clicar os 4 cantos.
    Se force_manual=False, tenta detecção automática e só pede cliques se falhar.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Imagem não encontrada: {image_path}")

    centers = find_shot_centroids(img)

    corners = None
    if not force_manual:
        corners = detect_court_corners(img)

    if force_manual or corners is None:
        print("Por favor, clique os 4 cantos da quadra na imagem.")
        pts = get_corners_from_click(img)
        if pts is None:
            raise RuntimeError("Operação cancelada ou pontos insuficientes.")
        corners = pts

    src = order_pts(corners.astype('float32'))
    H = compute_homography(src)
    coords = pixels_to_court_coords(centers, H)
    df = pd.DataFrame(coords, columns=['X','Y'])
    return df

# Conversão de png em dataframe
df = image_to_shot_dataframe(r"D:\PyCharm 2025.3\Projeto_final_EBAC\PNGs\BSB\VFLA2_L_Certo.png")
df.head(10)

Por favor, clique os 4 cantos da quadra na imagem.
Clique 4 pontos na ordem: TL, TR, BR, BL.
Botão esquerdo: adicionar; Botão direito: desfazer; ESC: cancelar


,X,Y
0,15.606792,47.801964
1,15.606705,47.036148
2,13.034245,33.446236
3,-19.661289,19.666252
4,22.161457,17.157440
5,-9.250635,12.742764
6,-21.874857,8.310623
7,-22.593124,7.347317
8,23.089455,7.968414
9,-21.269426,5.038582


In [92]:
'''
filename = "shots.csv"
# salva pela primeira vez (cria o arquivo)
df.to_csv(filename, index=False)
# depois, para acrescentar df_novo ao mesmo arquivo:
df_novo.to_csv(filename, mode='a', header=False, index=False)
'''


filename = "D:\\PyCharm 2025.3\\Projeto_final_EBAC\\Shot_Charts\\BSB_Certos.csv"
# salva o dataframe como arquivo
df.to_csv(filename, index=False, header = False, mode = 'a')